# Resume Screening & Ranking System
### Future Interns - Machine Learning Task 3

**Goal:** Build an NLP-based system that reads resumes, extracts skills, compares them to a job description, and ranks candidates with a clear, plain-English explanation of *why* - the kind of output a recruiter or HR manager could actually use.

**Dataset:** [Resume Dataset (Kaggle)](https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset) - 2,484 resumes across 24 job categories, with raw resume text and a category label for each.

**Approach, in one sentence:** clean the text -> extract skills with a hybrid of a curated skill list (precise) and spaCy noun phrases (exploratory) -> convert resumes and a job description into TF-IDF vectors -> rank by a weighted combination of overall content similarity and concrete skill-list overlap -> explain every score in plain English.

This notebook covers the core CSV-based pipeline. A separate script (`score_pdf_resumes.py`) applies the same pipeline to real, individual PDF resumes, with its own honest caveats about data overlap - see the **Conclusion** section for a summary of those results.


In [94]:
import pandas as pd
import re
import nltk
import spacy
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

pd.set_option('display.max_colwidth', 120)
print("Libraries loaded.")


Libraries loaded.


## Step 1-2: Load and Inspect the Data

Before writing any processing logic, it's worth understanding exactly what we're working with: how many resumes, what columns are available, and what job categories exist. This shapes every decision that follows.


In [95]:
# Update this path to wherever Resume.csv lives on your machine
DATA_PATH = r"C:\Users\nkonj\OneDrive\Desktop\Resume Screening\Resume\Resume.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nCategories and resume counts:")
print(df['Category'].value_counts())


Shape: (2484, 4)

Columns: ['ID', 'Resume_str', 'Resume_html', 'Category']

Categories and resume counts:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


**What this tells us:** the dataset has `Resume_str` (the raw resume text we'll work with), `Resume_html` (formatting we don't need), and `Category` (a label for the job field the resume is filed under). Categories range from 22 resumes (BPO) to 120 (Information Technology, Business Development) - useful to know since smaller categories will have noisier statistics.

We also found, while testing this dataset, that **one resume (ID 12632728) is blank/whitespace-only** - a genuine data quality issue in the source file, not something our pipeline causes. We drop it below.


In [96]:
# Drop the one blank resume found during testing
blank_mask = df['Resume_str'].isna() | (df['Resume_str'].str.strip() == '')
print(f"Dropping {blank_mask.sum()} blank resume(s): {df.loc[blank_mask, 'ID'].tolist()}")
df = df[~blank_mask].copy()
print("New shape:", df.shape)


Dropping 1 blank resume(s): [12632728]
New shape: (2483, 4)


## Step 3: Text Cleaning

Raw resume text is messy in predictable ways, and each cleaning step exists to fix a specific problem:

| Problem | Why it hurts us | Fix |
|---|---|---|
| Mixed case ("Python" vs "python") | Treated as different words otherwise | Lowercase everything |
| Punctuation, special characters | Confuses word matching | Strip to letters/numbers/spaces |
| Non-breaking spaces (`\xa0`) | Invisible, but not a real space character | Replace with regular spaces |
| Stopwords ("the", "and", "with") | Appear in every resume, add no signal | Remove for general text analysis |
| Extra whitespace | Formatting artifacts from source data | Collapse to single spaces |

One deliberate design choice: we keep **two cleaned versions** of each resume going forward. One has stopwords removed (for general text statistics), and one keeps stopwords (for skill-list matching, where multi-word phrases like "project management" need their exact word sequence intact - removing "of" from "knowledge of networking" wouldn't break this example, but it's safer in general to preserve phrase structure for matching).


In [97]:
def clean_resume_text(text):
    """
    Cleans raw resume text: lowercase, strip punctuation, collapse whitespace,
    and remove stopwords. Used for the general TF-IDF similarity comparison.
    """
    if not isinstance(text, str):
        return ''
    text = text.replace('\xa0', ' ')
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [w for w in words if w not in STOPWORDS]
    return ' '.join(words)


def prep_for_skill_matching(text):
    """
    A lighter cleaning pass that KEEPS stopwords, so multi-word skill
    phrases like 'windows server' or 'project management' stay intact.
    Also keeps '#' and '+' so 'c#' and 'c++' remain detectable.
    """
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s#+]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df['Resume_clean'] = df['Resume_str'].apply(clean_resume_text)
df['Resume_clean_phrases'] = df['Resume_str'].apply(prep_for_skill_matching)

# Before / after comparison on one resume
sample_idx = df[df['Category'] == 'INFORMATION-TECHNOLOGY'].index[0]
print("BEFORE:\n", df.loc[sample_idx, 'Resume_str'][:300])
print("\nAFTER (stopwords removed):\n", df.loc[sample_idx, 'Resume_clean'][:300])


BEFORE:
          INFORMATION TECHNOLOGY         Summary     Dedicated  Information Assurance Professional  well-versed in analyzing and mitigating risk and finding cost-effective solutions. Excels at boosting performance and productivity by establishing realistic goals and enforcing deadlines.  Versatile IT

AFTER (stopwords removed):
 information technology summary dedicated information assurance professional well versed analyzing mitigating risk finding cost effective solutions excels boosting performance productivity establishing realistic goals enforcing deadlines versatile professional 37 years enterprise design engineering m


In [98]:
# Quick sanity check: cleaning shouldn't drastically distort the data
avg_before = df['Resume_str'].apply(lambda x: len(str(x).split())).mean()
avg_after = df['Resume_clean'].apply(lambda x: len(x.split())).mean()
print(f"Average word count before cleaning: {avg_before:.1f}")
print(f"Average word count after cleaning:  {avg_after:.1f}")
print(f"(Stopword removal cut word count by about {100 * (1 - avg_after/avg_before):.0f}%, as expected)")


Average word count before cleaning: 811.7
Average word count after cleaning:  622.7
(Stopword removal cut word count by about 23%, as expected)


## Step 4: Skill Extraction (Hybrid Approach)

We use two complementary techniques:

**A. Predefined skill lists (per category).** Precise and explainable - if a resume matches "Python", we can say exactly why it matched. The tradeoff: a fixed list can miss skills it doesn't know to look for.

**B. spaCy noun phrase extraction.** spaCy understands grammar - it can identify noun phrases (e.g. "customer service", "database administration") automatically, without needing a predefined list. This is noisier (it surfaces some non-skills too) but catches things a fixed list misses. We treat this as a bonus "things we noticed" layer rather than part of the core score.

### Building the skill lists: an evidence-based process

Rather than guessing what skills matter, each list below was built the same way: start with domain knowledge, then validate against **actual word frequency counts** in this dataset, and adjust. For example, our first pass at the IT skill list (and Agriculture, Healthcare, Sales) initially produced uneven results - some categories matched well, others poorly - until we checked what words real resumes in those categories actually used and added evidence-based terms. We document a couple of these corrections directly in the comments below, since "the list was strong from the start" was not actually true for every category, and that revision process is worth being transparent about.


In [99]:
SKILLS_BY_CATEGORY = {
    "INFORMATION-TECHNOLOGY": [
        "python", "java", "c++", "c#", "sql", "javascript", "html", "css",
        "linux", "windows server", "active directory", "networking",
        "cloud computing", "aws", "azure", "cybersecurity", "security",
        "database", "api", "devops", "git", "agile", "scrum", "troubleshooting",
        "hardware", "software", "server", "microsoft", "application", "systems",
        "data",
    ],
    "AGRICULTURE": [
        "farming", "crop production", "livestock", "irrigation", "pesticide",
        "herbicide", "soil management", "harvesting", "agronomy", "greenhouse",
        "equipment operation", "tractor", "planting", "fertilizer",
        "pest control", "quarantine", "plant protection", "organic farming",
        "sustainability", "gps mapping", "yield", "crop rotation",
        "animal husbandry", "food safety", "usda", "research",
        "agricultural science", "environmental science", "regulatory compliance",
        "field research", "data collection", "laboratory", "extension services",
        "community outreach",
        # Evidence-based additions after checking real word frequencies -
        # the initial list scored 8/63 Agriculture resumes at zero matches,
        # mostly resumes using inspection/education/compliance language
        # rather than hands-on farming terms
        "inspection", "grain", "compliance", "curriculum", "biology",
        "environmental",
    ],
    "HEALTHCARE": [
        "patient care", "nursing", "clinical", "vital signs",
        "medication administration", "charting", "ehr",
        "electronic health records", "hipaa", "triage", "cpr", "bls", "acls",
        "phlebotomy", "patient assessment", "care plan", "infection control",
        "medical terminology", "scheduling", "insurance verification",
        "customer service", "registration", "icd-10", "cpt",
        "patient registration", "medical records", "case management",
        "client relations",
        # Evidence-based additions - the initial list was biased toward
        # multi-word phrases and missed common single-word terms
        # ("patient", "medical", "care") that appear extremely frequently
        "patient", "medical", "healthcare", "health", "care", "physician",
    ],
    "SALES": [
        "cold calling", "lead generation", "crm", "salesforce",
        "account management", "negotiation", "closing", "upselling",
        "cross-selling", "pipeline management", "quota", "prospecting",
        "customer relationship", "b2b", "b2c", "territory management",
        "sales forecasting", "presentation skills", "client retention",
        "contract negotiation", "retail", "inventory management",
        "merchandising", "point of sale", "cash handling", "store operations",
        "customer service",
        # Evidence-based additions, same reasoning as Healthcare above
        "sales", "customer", "product", "inventory", "marketing", "cash",
    ],
    "BUSINESS-DEVELOPMENT": [
        "market research", "strategic partnerships", "business strategy",
        "stakeholder management", "negotiation", "client acquisition",
        "revenue growth", "market analysis", "competitive analysis",
        "proposal writing", "networking", "relationship building",
        "project management", "budget management", "kpi", "roi",
        "business plan", "vendor management", "marketing",
        "account management", "client relationships", "business development",
        "strategic planning",
    ],
}

print("Skill lists defined for", len(SKILLS_BY_CATEGORY), "categories")
for cat, skills in SKILLS_BY_CATEGORY.items():
    print(f"  {cat}: {len(skills)} skills")


Skill lists defined for 5 categories
  INFORMATION-TECHNOLOGY: 31 skills
  AGRICULTURE: 40 skills
  HEALTHCARE: 34 skills
  SALES: 33 skills
  BUSINESS-DEVELOPMENT: 23 skills


In [100]:
def match_skills(text, category, skills_by_category=SKILLS_BY_CATEGORY):
    """Returns the list of skills (from this resume's own category list) found in the text."""
    skill_list = skills_by_category.get(category, [])
    if not skill_list or not text:
        return []
    return [skill for skill in skill_list if skill in text]


df['matched_skills'] = df.apply(
    lambda row: match_skills(row['Resume_clean_phrases'], row['Category']),
    axis=1,
)
df['matched_skills_count'] = df['matched_skills'].apply(len)

print("Average matched skills per resume, by covered category:")
for category in SKILLS_BY_CATEGORY:
    subset = df[df['Category'] == category]
    avg = subset['matched_skills_count'].mean()
    zero_count = (subset['matched_skills_count'] == 0).sum()
    print(f"  {category:25s} avg={avg:5.2f}   zero-match={zero_count}/{len(subset)}")


Average matched skills per resume, by covered category:
  INFORMATION-TECHNOLOGY    avg=10.07   zero-match=0/120
  AGRICULTURE               avg= 3.40   zero-match=3/63
  HEALTHCARE                avg= 8.31   zero-match=0/115
  SALES                     avg= 7.16   zero-match=0/116
  BUSINESS-DEVELOPMENT      avg= 4.25   zero-match=0/119


**Result:** all 5 covered categories now have healthy, comparable skill-match rates (low or zero "zero-match" counts), thanks to the evidence-based tuning above. Resumes outside these 5 categories will show 0 matched skills - that's expected, not a bug, since we haven't built lists for those fields yet (a clear path for future extension, using the same evidence-based process).


In [101]:
print("Resumes outside our 5 covered categories:",
      (~df['Category'].isin(SKILLS_BY_CATEGORY.keys())).sum(),
      "out of", len(df))


Resumes outside our 5 covered categories: 1950 out of 2483


### Bonus layer: spaCy noun phrase extraction

This step processes every resume's grammar to pull out noun phrases - it's slower than skill-list matching (spaCy has to parse sentence structure for each resume), so we show it on a few examples here rather than running it across all 2,483 resumes in this notebook. The full multi-category skill extraction script (`extract_skills.py`) runs this across the entire dataset and saves the results.


In [102]:
print("Loading spaCy model (only needs to run once)...")
nlp = spacy.load("en_core_web_sm")

GENERIC_NOISE = {
    'years', 'year', 'experience', 'company', 'companyname', 'team',
    'role', 'work', 'job', 'time', 'environment', 'ability', 'skills',
}

def extract_noun_phrases(raw_text, max_phrases=10):
    if not isinstance(raw_text, str) or not raw_text.strip():
        return []
    doc = nlp(raw_text[:5000])
    phrases, seen = [], set()
    for chunk in doc.noun_chunks:
        phrase = chunk.text.lower().strip()
        word_count = len(phrase.split())
        if (1 <= word_count <= 3 and phrase not in seen
                and phrase not in GENERIC_NOISE and len(phrase) > 2):
            phrases.append(phrase)
            seen.add(phrase)
    return phrases[:max_phrases]


sample_it = df[df['Category'] == 'INFORMATION-TECHNOLOGY'].iloc[0]
print("Matched skills (from list):", sample_it['matched_skills'])
print("\nNoun phrases (spaCy, bonus layer):", extract_noun_phrases(sample_it['Resume_str']))


Loading spaCy model (only needs to run once)...
Matched skills (from list): ['html', 'linux', 'windows server', 'active directory', 'security', 'troubleshooting', 'hardware', 'software', 'server', 'application', 'systems', 'data']

Noun phrases (spaCy, bonus layer): ['information technology         summary', 'information assurance professional', 'risk', 'cost-effective solutions', 'excels', 'performance', 'productivity', 'realistic goals', 'deadlines', 'versatile it']


## Step 5: Job Description Parsing & Similarity Scoring

This is where the system becomes a genuine *screening* tool: comparing resumes against a target job description.

### The core idea: TF-IDF + Cosine Similarity

**TF-IDF** (Term Frequency - Inverse Document Frequency) converts text into numbers in a smart way. Words that are frequent *in one document* but rare *across all documents* get the highest weight - so "Kubernetes" matters far more than "team" for telling resumes apart, even though "team" might appear more often in any single resume.

**Cosine similarity** then measures the angle between two TF-IDF vectors, not their length. This means a short job description and a long resume can be compared fairly - similarity reflects how much the *content emphasis* overlaps, not how many words either document has.

### Combining two signals

We use two scores together:
1. **Similarity score** (TF-IDF + cosine) - overall content/contextual fit
2. **Skill match score** - precise checklist overlap, using the skill lists from Step 4

**Final score = 0.7 x similarity + 0.3 x skill match.** This weighting means overall content fit dominates the ranking, with the skill checklist acting as a meaningful but secondary adjustment. It's a deliberate, explainable design choice - not the only valid one, but one we can fully justify.

We validate this approach across **all 5 categories** we built skill lists for for, to confirm the system generalizes rather than only working for our original IT demo.


In [ ]:
JOB_DESCRIPTIONS = {
    "INFORMATION-TECHNOLOGY": {
        "title": "IT Systems Administrator",
        "text": '''
IT Systems Administrator

We are seeking an experienced IT Systems Administrator to manage and support
our growing technology infrastructure. The ideal candidate will have strong
hands-on experience with Windows Server environments, networking, and
enterprise security practices, and will be comfortable working directly with
end users to resolve technical issues.

Responsibilities:
- Administer and maintain Windows Server, Active Directory, and network
  infrastructure across multiple office locations
- Provide technical support and troubleshooting for hardware, software, and
  network issues
- Manage user accounts, permissions, and security policies
- Support cloud infrastructure on AWS or Azure
- Maintain database systems and ensure data integrity
- Collaborate with the development team on application deployment and
  DevOps practices

Requirements:
- 3+ years of experience in IT systems administration or technical support
- Strong knowledge of Windows Server, Active Directory, and networking
- Experience with cloud platforms (AWS or Azure)
- Familiarity with database management (SQL)
- Understanding of cybersecurity best practices
- Experience with scripting or programming (Python preferred)
- Experience with Agile workflows and tools like Jira is a plus
''',
    },
    "AGRICULTURE": {
        "title": "Agricultural Field Technician",
        "text": '''
Agricultural Field Technician

We are looking for a detail-oriented Agricultural Field Technician to conduct
field surveys, identify plant pests, and support agricultural research and
regulatory compliance efforts.

Responsibilities:
- Conduct ground-based visual surveys for invasive insects and plant pests
- Identify plant hosts and pest species accurately
- Collect and record field data using GPS and standard survey equipment
- Implement plant protection standards to safeguard agriculture and natural
  resources
- Support laboratory analysis and data collection efforts
- Maintain compliance with USDA and regulatory standards

Requirements:
- Background in environmental biology, agricultural science, or related field
- Experience with field research and data collection
- Familiarity with regulatory compliance standards
- Strong attention to detail and organizational skills
- Comfortable working outdoors in varied conditions
''',
    },
    "HEALTHCARE": {
        "title": "Registered Nurse - Medical/Surgical Unit",
        "text": '''
Registered Nurse - Medical/Surgical Unit

We are seeking a compassionate, detail-oriented Registered Nurse to join our
Medical/Surgical unit. The ideal candidate will have strong clinical
assessment skills and experience providing patient-centered care in a
fast-paced hospital environment.

Responsibilities:
- Provide direct patient care including assessments, medication
  administration, and wound care
- Monitor and document patient vital signs and progress
- Collaborate with physicians and the care team on treatment plans
- Maintain accurate patient charting and electronic health records
- Communicate effectively with patients and families regarding care plans
- Follow infection control and best practice protocols

Requirements:
- Active RN license, BSN preferred
- Strong organizational and communication skills
- Experience with electronic medical records
- CPR/BLS certification
''',
    },
    "SALES": {
        "title": "Retail Sales Associate",
        "text": '''
Retail Sales Associate

We are seeking an energetic Retail Sales Associate to drive store sales,
deliver excellent customer service, and support day-to-day store operations.

Responsibilities:
- Greet and assist customers, identifying their needs and recommending
  products
- Process transactions accurately using point of sale systems
- Maintain merchandising standards and store appearance
- Support inventory management and stock replenishment
- Build customer relationships to drive repeat business and upselling
- Meet individual and store sales targets

Requirements:
- Previous retail or customer service experience preferred
- Strong communication and interpersonal skills
- Comfortable with cash handling and point of sale systems
- Ability to work in a fast-paced environment
- Basic understanding of sales and marketing principles
''',
    },
    "BUSINESS-DEVELOPMENT": {
        "title": "Business Development Manager",
        "text": '''
Business Development Manager

We are seeking a strategic Business Development Manager to identify growth
opportunities, build partnerships, and drive revenue through new client
acquisition and relationship management.

Responsibilities:
- Conduct market research to identify new business opportunities
- Build and maintain strategic partnerships and client relationships
- Develop and present business proposals to prospective clients
- Manage the sales pipeline and forecast revenue growth
- Collaborate with marketing on go-to-market strategy
- Track KPIs and report on business development performance

Requirements:
- Proven experience in business development, sales, or strategic partnerships
- Strong negotiation and relationship-building skills
- Experience with market and competitive analysis
- Strategic planning and project management skills
- Excellent communication and presentation skills
''',
    },
}

print(f"Defined {len(JOB_DESCRIPTIONS)} job descriptions, one per covered category")


Defined 5 job descriptions, one pe covered category


In [115]:
SIMILARITY_WEIGHT = 0.7
SKILL_WEIGHT = 0.3

def score_resumes_for_category(df, category, job_title, job_text, skills_by_category=SKILLS_BY_CATEGORY):
    """
    Computes similarity_score, skill_match_score, and final_score for every
    resume in the dataset against one job description. Returns the full
    dataframe with these columns added (computed for ALL resumes, so we can
    also sanity-check that the target category scores highest on average).
    """
    df = df.copy()
    jd_clean = clean_resume_text(job_text)

    # Fit TF-IDF jointly on all resumes + the job description so they
    # share one vocabulary space - required for cosine similarity to be
    # meaningful
    corpus = df['Resume_clean'].tolist() + [jd_clean]
    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(corpus)
    df['similarity_score'] = cosine_similarity(tfidf_matrix[:-1], tfidf_matrix[-1]).flatten()

    # Figure out which skills from this category's list actually appear
    # in the job description text itself
    jd_prepped = prep_for_skill_matching(job_text)
    skill_list = skills_by_category.get(category, [])
    required_skills = sorted(set(s for s in skill_list if s in jd_prepped))

    df['matched_required_skills'] = df['Resume_clean_phrases'].apply(
        lambda t: [s for s in required_skills if s in t]
    )
    df['skill_match_score'] = df['matched_required_skills'].apply(
        lambda m: len(m) / len(required_skills) if required_skills else 0
    )

    df['final_score'] = (
        SIMILARITY_WEIGHT * df['similarity_score']
        + SKILL_WEIGHT * df['skill_match_score']
    )

    return df, required_skills


print("Scoring function defined.")


Scoring function defined.


In [109]:
# Run the scoring + sanity check across all 5 categories
print(f"{'Category':25s} {'Own-cat avg sim':>16s} {'Top category':>25s} {'Match?':>8s}")
print("-" * 80)

scored_results = {}
required_skills_by_cat = {}

for category, jd in JOB_DESCRIPTIONS.items():
    scored_df, required_skills = score_resumes_for_category(
        df, category, jd['title'], jd['text']
    )
    scored_results[category] = scored_df
    required_skills_by_cat[category] = required_skills

    cat_avgs = scored_df.groupby('Category')['similarity_score'].mean().sort_values(ascending=False)
    own_avg = cat_avgs.get(category, 0)
    top_cat = cat_avgs.index[0]
    match = "YES" if top_cat == category else "NO"
    print(f"{category:25s} {own_avg:16.4f} {top_cat:>25s} {match:>8s}")


Category                   Own-cat avg sim              Top category   Match?
--------------------------------------------------------------------------------
INFORMATION-TECHNOLOGY              0.1916    INFORMATION-TECHNOLOGY      YES
AGRICULTURE                         0.0851               AGRICULTURE      YES
HEALTHCARE                          0.1525                HEALTHCARE      YES
SALES                               0.1762                     SALES      YES
BUSINESS-DEVELOPMENT                0.1930      BUSINESS-DEVELOPMENT      YES


**Validation result:** for every one of the 5 categories, resumes from that *exact* category score highest on average against their matching job description - without ever telling the system which category a resume belongs to. The system is genuinely learning from content, not just category labels. This is the strongest evidence we have that the approach generalizes beyond our original IT demo.


## Step 6: Candidate Ranking & Skill Gap Reports

This is the actual deliverable a recruiter would read: a ranked shortlist with a plain-English explanation for each candidate - what they have, what they're missing, and why they ranked where they did.


In [106]:
def explain_match(row, required_skills):
    matched = row['matched_required_skills']
    missing = [s for s in required_skills if s not in matched]
    pct = int(round(row['skill_match_score'] * 100))
    explanation = f"Matched {len(matched)} of {len(required_skills)} required skills ({pct}%). "
    if matched:
        explanation += f"Strengths: {', '.join(matched)}. "
    if missing:
        explanation += f"Missing: {', '.join(missing)}."
    else:
        explanation += "No required skills missing."
    return explanation


def print_top_candidates(category, n=5):
    scored_df = scored_results[category]
    required_skills = required_skills_by_cat[category]
    job_title = JOB_DESCRIPTIONS[category]['title']

    target_df = scored_df[scored_df['Category'] == category].sort_values(
        'final_score', ascending=False
    )

    print(f"=== Top {n} candidates for: {job_title} ({category}) ===\n")
    for rank, (idx, row) in enumerate(target_df.head(n).iterrows(), start=1):
        explanation = explain_match(row, required_skills)
        print(f"#{rank} - Resume ID {row['ID']}  |  Final Score: {row['final_score']:.3f}  "
              f"(similarity={row['similarity_score']:.3f}, skill_match={row['skill_match_score']:.2f})")
        print(f"    {explanation}\n")


print_top_candidates("INFORMATION-TECHNOLOGY", n=5)


=== Top 5 candidates for: IT Systems Administrator (INFORMATION-TECHNOLOGY) ===

#1 - Resume ID 20824105  |  Final Score: 0.439  (similarity=0.379, skill_match=0.58)
    Matched 11 of 19 required skills (58%). Strengths: active directory, application, aws, data, networking, python, security, server, software, systems, windows server. Missing: agile, azure, cybersecurity, database, devops, hardware, sql, troubleshooting.

#2 - Resume ID 39413067  |  Final Score: 0.431  (similarity=0.344, skill_match=0.63)
    Matched 12 of 19 required skills (63%). Strengths: active directory, application, data, database, hardware, security, server, software, sql, systems, troubleshooting, windows server. Missing: agile, aws, azure, cybersecurity, devops, networking, python.

#3 - Resume ID 27295996  |  Final Score: 0.422  (similarity=0.355, skill_match=0.58)
    Matched 11 of 19 required skills (58%). Strengths: active directory, application, azure, data, database, hardware, security, server, software,

In [107]:
# The same report works for every category we've covered - here's Healthcare
print_top_candidates("HEALTHCARE", n=3)


=== Top 3 candidates for: Registered Nurse - Medical/Surgical Unit (HEALTHCARE) ===

#1 - Resume ID 25328428  |  Final Score: 0.518  (similarity=0.446, skill_match=0.69)
    Matched 11 of 16 required skills (69%). Strengths: bls, care, charting, clinical, cpr, health, infection control, medical, patient, patient care, vital signs. Missing: care plan, electronic health records, medical records, medication administration, physician.

#2 - Resume ID 46349752  |  Final Score: 0.518  (similarity=0.471, skill_match=0.62)
    Matched 10 of 16 required skills (62%). Strengths: care, clinical, cpr, health, infection control, medical, medication administration, patient, patient care, vital signs. Missing: bls, care plan, charting, electronic health records, medical records, physician.

#3 - Resume ID 25834360  |  Final Score: 0.515  (similarity=0.468, skill_match=0.62)
    Matched 10 of 16 required skills (62%). Strengths: care, care plan, clinical, health, medical, medical records, medication a

In [108]:
# And Agriculture - notice fewer required skills get matched here, since
# this job description (Agricultural Field Technician) leans toward
# research/lab language rather than hands-on farming terms. A resume that's
# strong in farming specifically may score lower against THIS particular
# posting - which reflects a real distinction between job types within the
# same broad category, not a flaw in the matching logic.
print_top_candidates("AGRICULTURE", n=3)


=== Top 3 candidates for: Agricultural Field Technician (AGRICULTURE) ===

#1 - Resume ID 11197262  |  Final Score: 0.383  (similarity=0.313, skill_match=0.55)
    Matched 6 of 11 required skills (55%). Strengths: biology, data collection, environmental, laboratory, plant protection, research. Missing: agricultural science, compliance, field research, regulatory compliance, usda.

#2 - Resume ID 18242317  |  Final Score: 0.379  (similarity=0.308, skill_match=0.55)
    Matched 6 of 11 required skills (55%). Strengths: biology, data collection, environmental, laboratory, plant protection, research. Missing: agricultural science, compliance, field research, regulatory compliance, usda.

#3 - Resume ID 15603319  |  Final Score: 0.374  (similarity=0.301, skill_match=0.55)
    Matched 6 of 11 required skills (55%). Strengths: biology, data collection, environmental, laboratory, plant protection, research. Missing: agricultural science, compliance, field research, regulatory compliance, usda.

## Honest Limitations

A system like this is only trustworthy if its limitations are stated plainly. Here's what we found through testing, not just what we assume could go wrong:

**1. TF-IDF measures word overlap, not meaning.** While testing this pipeline against real, individual PDF resumes (see `score_pdf_resumes.py`), a senior IT architect's resume scored *lower* against a generic "IT Systems Administrator" posting than a nurse's resume scored against a nursing posting - even though the architect's resume was clearly, deeply technical. The reason: senior/specialized vocabulary ("OLTP", "HTAP", "operational data store") doesn't lexically overlap much with more operational job-posting language, even when the roles are conceptually related. A semantic/embedding-based approach would likely handle this better, at the cost of being less explainable.

**2. Skill lists only cover 5 of 24 categories.** Resumes from the other 19 categories (Chef, Teacher, Finance, Engineering, and others) currently show 0 matched skills, since we haven't built evidence-based skill lists for them yet. This is a clear, scoped next step rather than a hidden gap.

**3. Skill-list coverage depends on how the job description itself is worded.** The same skill list can produce very different "required skills" sets depending on the specific job posting - we saw this directly with Agriculture, where a research-leaning posting matched far fewer of a hands-on farmer's actual skills than a more general one would. This is arguably correct behavior (different job postings genuinely require different things) but is worth knowing when interpreting scores.

**4. PDF text extraction can merge words together.** Resumes extracted from certain PDF sources (we found this with one of our real-world PDF test files) sometimes lose the spaces between words during extraction - e.g. "EnvironmentalBiologistand" instead of "Environmental Biologist and." This can cause skill matching to undercount, since a merged word won't match a clean skill-list entry. `score_pdf_resumes.py` includes a basic detector that flags resumes likely affected by this.

**5. One blank source record.** Resume ID 12632728 was empty/whitespace-only in the original dataset - a source data quality issue, not a pipeline bug. We excluded it.

**6. Single-category design.** Each resume is currently scored only against its *own* dataset category's skill list. A more flexible production system would let a recruiter score any resume against any job description, regardless of the resume's original category label.


## Conclusion

### What this system does
Given a job description, it ranks candidates by a combination of overall content fit (TF-IDF + cosine similarity) and concrete skill checklist overlap, and explains every score in plain English - matched skills, missing skills, and why.

### Validated across 5 categories
For Information Technology, Agriculture, Healthcare, Sales, and Business Development, resumes from the correct category consistently scored highest against their matching job description - without the system ever being told the category directly. This is real evidence the approach generalizes, not just a single cherry-picked demo.

### Tested against real-world PDF resumes
We also ran this pipeline against 4 individual PDF resumes (not the training CSV) using `score_pdf_resumes.py`. One important, transparent finding from that testing: **all 4 PDFs turned out to already exist inside `Resume.csv`** (all mislabeled under the Agriculture category in the source data) - meaning they were not truly held-out, unseen data. We're reporting this honestly rather than presenting those results as a clean blind test. A genuine next step would be testing against resumes confirmed to be outside the training corpus entirely.

### Mapping back to the task brief

| Requirement | Status |
|---|---|
| Resume text cleaning & preprocessing | Done |
| Skill extraction using NLP | Done (5 categories, evidence-based) |
| Job description parsing | Done |
| Resume-to-role similarity scoring | Done |
| Candidate ranking based on role fit | Done |
| Skill gap identification | Done |
| Clear explanation for non-technical users | Done (plain-English explanation per candidate) |
| Tested on real-world PDF resumes | Done, with documented caveats |

### Possible future improvements
- Extend skill lists to the remaining 19 categories using the same evidence-based process
- Try a semantic/embedding-based similarity measure alongside TF-IDF to address the vocabulary-mismatch limitation
- Build a simple interface so a recruiter can paste in any job description and get ranked results without editing code
- Source genuinely external test resumes for a true held-out validation set
